## 1 — Imports & Setup

In [ ]:
import os, sys, time, gc, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import wfdb
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
print(f'Python {sys.version.split()[0]}, numpy {np.__version__}, wfdb {wfdb.__version__}')

In [ ]:
# TensorFlow setup
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv1D, Dense, Dropout, Add, Activation, BatchNormalization
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Abort immediately if no GPU is visible. A 21-patient search that silently
# falls back to CPU runs 10-30x slower - days instead of hours - and there is
# no sign of it until the job is already burning allocation.
REQUIRE_GPU = True

# Which GPU(s) to use. None = all visible. On a shared node prefer setting
# CUDA_VISIBLE_DEVICES in the submit script rather than changing this.
GPU_INDEX = None

gpus = tf.config.list_physical_devices('GPU')

if GPU_INDEX is not None and gpus:
    tf.config.set_visible_devices(gpus[GPU_INDEX], 'GPU')
    gpus = tf.config.list_physical_devices('GPU')

# Grow GPU memory on demand - stops one process reserving a whole card on a
# shared node. Must be set before any tensor is allocated.
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(f'  memory growth not set for {gpu.name}: {e}')

build = dict(tf.sysconfig.get_build_info())
print(f'TensorFlow  : {tf.__version__}')
print(f'CUDA build  : {build.get("is_cuda_build")}   '
      f'(cuda {build.get("cuda_version")}, cudnn {build.get("cudnn_version")})')
print(f'GPUs visible: {len(gpus)}')
for i, gpu in enumerate(gpus):
    try:
        det = tf.config.experimental.get_device_details(gpu)
        print(f'  [{i}] {det.get("device_name", gpu.name)} '
              f'(compute capability {det.get("compute_capability")})')
    except Exception:
        print(f'  [{i}] {gpu.name}')

## 2 — Constants & Configuration

In [ ]:
# Dataset path (LOCAL / OFFLINE)
DATA_DIR = r"E:\NU\Tasks\Offline Task 1\mit-bih-arrhythmia-database-1.0.0\mit-bih-arrhythmia-database-1.0.0"

# Output directory (created next to this notebook)
OUT_DIR = os.path.join(os.getcwd(), 'mogwo21_output')
os.makedirs(OUT_DIR, exist_ok=True)

# Paper constants
FS           = 360
TOTAL_STEPS  = 100_000
TRAIN_STEPS  = 40_000
VAL_STEPS    = 10_000
TEST_STEPS   = 50_000
LOOKBACK     = 10

# Patients
ALL_PATIENTS = [
    '100', '101', '102', '103', '104', '105',
    '106', '107', '108', '109', '112', '113',
    '114', '115', '116', '117', '119',
    '121', '122', '123', '124'
]
assert len(ALL_PATIENTS) == 21

# Patients withheld from the search. Empty = search on all 21, as advised.
# Putting e.g. ['104', '114', '122'] here gives a held-out set the optimizer
# never sees, which pre-empts the "you tuned on your test set" objection.
# See the note in Section 2.1.
HOLDOUT_PATIENTS = []

PILOT_PATIENTS = [p for p in ALL_PATIENTS if p not in HOLDOUT_PATIENTS]
PILOT_HORIZON  = 10

# Fixed hyperparameters (literature-justified)
FIXED_N_BLOCKS    = 3    # RF = 29 >> lookback = 10
FIXED_KERNEL_SIZE = 3    # Bai et al. default
FIXED_BATCH_SIZE  = 128  # Low sensitivity

# Training budget per evaluation (unchanged, keeps runs comparable)
EVAL_EPOCHS   = 100
EVAL_PATIENCE = 20

# MOGWO configuration
POP_SIZE     = 5     # wolves per iteration
MAX_ITER     = 5     # MOGWO iterations
ARCHIVE_MAX  = 20    # max Pareto archive size
N_GRIDS      = 10    # grid divisions per objective

# Post-hoc selection weights (TA-specified)
W_RMSE = 0.70
W_TIME = 0.30
assert abs(W_RMSE + W_TIME - 1.0) < 1e-9

# Search bounds
FILTER_OPTIONS = [32, 64, 128, 256]
LB = np.array([0.0, 0.05, -4.0])   # [filter_idx, dropout, log10_lr]
UB = np.array([3.0, 0.40, -2.0])

# Reproducibility & resume
SEED       = 42
CACHE_PATH = os.path.join(OUT_DIR, 'mogwo21_eval_cache.json')

np.random.seed(SEED)
tf.random.set_seed(SEED)

# Runtime estimate, from the measured 5-patient run
SEC_PER_PATIENT_EST = 158.0     # mean of 29 evals on 2xT4
N_EVALS_PLANNED     = POP_SIZE * (1 + MAX_ITER)
est_hours = SEC_PER_PATIENT_EST * len(PILOT_PATIENTS) * N_EVALS_PLANNED / 3600

print(f'Data dir       : {DATA_DIR}')
print(f'  exists       : {os.path.isdir(DATA_DIR)}')
print(f'Output dir     : {OUT_DIR}')
print(f'MOGWO config   : {POP_SIZE} wolves x (1 init + {MAX_ITER} iters) = {N_EVALS_PLANNED} evals')
print(f'Pilot patients : {len(PILOT_PATIENTS)} -> {PILOT_PATIENTS}')
print(f'Held out       : {HOLDOUT_PATIENTS if HOLDOUT_PATIENTS else "none"}')
print(f'Horizon        : H={PILOT_HORIZON}')
print(f'Fixed          : n_blocks={FIXED_N_BLOCKS} (RF=29), kernel={FIXED_KERNEL_SIZE}, batch={FIXED_BATCH_SIZE}')
print(f'Search         : n_filters in {FILTER_OPTIONS}, dropout in [0.05, 0.40], lr in [1e-4, 1e-2]')
print(f'Objectives     : minimize RMSE, minimize training time')
print(f'Selection      : {W_RMSE:.2f} x RMSE_norm + {W_TIME:.2f} x time_norm (applied after the search)')
print(f'\nEstimated runtime at {SEC_PER_PATIENT_EST:.0f} s/patient/eval (2xT4 reference): {est_hours:.1f} h')

### 2.1 — Note on searching and reporting over the same patients

With `HOLDOUT_PATIENTS = []` the search set and the final evaluation set are the same 21 recordings. The hyperparameters that win are the ones that scored best on those recordings, and the paper then reports the score on those same recordings.

Part of that reported gain is genuinely better hyperparameters; part is those hyperparameters happening to suit these particular recordings. With no unseen patient left over, the two cannot be separated, and the reported RMSE is mildly optimistic. This is the standard protocol in this line of work, so it is defensible — but state it in the limitations section rather than leaving a reviewer to find it.

The cheap alternative is to set `HOLDOUT_PATIENTS` to 3–4 records, run the search on the remaining 17–18, and report the held-out patients separately as evidence that the configuration generalizes. Everything downstream in this notebook works either way.

## 3 — Data Pipeline

Identical preprocessing to the baseline — load the MLII lead, split, normalise (scaler fit on train only), then build multi-step sequences.

Sequences are built **once** here instead of inside every evaluation. At 21 patients × 30 evaluations that saves roughly half an hour of pure-Python looping; the vectorised `sliding_window_view` version replaces the original list-append loop and produces identical arrays.

In [ ]:
def load_ecg_signal(record_id, data_dir, n_steps=100_000):
    """Load the MLII lead from MIT-BIH, truncate/pad to n_steps."""
    path = os.path.join(data_dir, record_id)
    rec  = wfdb.rdrecord(path)
    sig_names_upper = [s.upper() for s in rec.sig_name]
    ch = sig_names_upper.index('MLII') if 'MLII' in sig_names_upper else 0
    signal = rec.p_signal[:, ch].astype(np.float32)
    if len(signal) < n_steps:
        pad = np.full(n_steps - len(signal), signal[-1], dtype=np.float32)
        signal = np.concatenate([signal, pad])
    return signal[:n_steps]


def preprocess_patient(signal, train_steps=40_000, val_steps=10_000):
    """Split into train/val/test, MinMax-normalise (fit on train only)."""
    train_end = train_steps
    val_end   = train_steps + val_steps
    train_raw = signal[:train_end]
    val_raw   = signal[train_end:val_end]
    test_raw  = signal[val_end:]
    scaler     = MinMaxScaler(feature_range=(0, 1))
    train_norm = scaler.fit_transform(train_raw.reshape(-1, 1)).flatten()
    val_norm   = scaler.transform(val_raw.reshape(-1, 1)).flatten()
    test_norm  = scaler.transform(test_raw.reshape(-1, 1)).flatten()
    return train_norm, val_norm, test_norm, scaler


def make_multistep_sequences(signal, lookback, horizon):
    """X = lookback window, y = next H steps. Vectorised; matches the original
    list-append loop element for element."""
    windows = sliding_window_view(np.asarray(signal, dtype=np.float32),
                                  lookback + horizon)
    X = np.ascontiguousarray(windows[:, :lookback])
    y = np.ascontiguousarray(windows[:, lookback:])
    return X, y


def compute_metrics(y_true, y_pred):
    yt, yp = y_true.flatten(), y_pred.flatten()
    return {
        'RMSE': float(np.sqrt(mean_squared_error(yt, yp))),
        'MAE':  float(mean_absolute_error(yt, yp)),
        'R2':   float(r2_score(yt, yp)),
    }


print('Data pipeline functions defined.')

In [ ]:
# Load pilot patients and precompute sequences once
patient_data = {}

print(f'Loading {len(PILOT_PATIENTS)} patients from local disk...\n')
t_load = time.time()

for rid in tqdm(PILOT_PATIENTS, desc='Patients'):
    signal = load_ecg_signal(rid, DATA_DIR, n_steps=TOTAL_STEPS)
    tr, vl, te, sc = preprocess_patient(signal, TRAIN_STEPS, VAL_STEPS)

    X_tr, y_tr = make_multistep_sequences(tr, LOOKBACK, PILOT_HORIZON)
    X_vl, y_vl = make_multistep_sequences(vl, LOOKBACK, PILOT_HORIZON)
    X_te, y_te = make_multistep_sequences(te, LOOKBACK, PILOT_HORIZON)

    patient_data[rid] = {
        'X_tr': X_tr.reshape(-1, LOOKBACK, 1), 'y_tr': y_tr,
        'X_vl': X_vl.reshape(-1, LOOKBACK, 1), 'y_vl': y_vl,
        'X_te': X_te.reshape(-1, LOOKBACK, 1), 'y_te': y_te,
        'scaler': sc,
    }
    tqdm.write(f'  Patient {rid:>3s} | train={len(X_tr):,}  val={len(X_vl):,}  test={len(X_te):,}')

mem_mb = sum(
    v[k].nbytes for v in patient_data.values()
    for k in ('X_tr', 'y_tr', 'X_vl', 'y_vl', 'X_te', 'y_te')
) / 1024**2

print(f'\n{len(patient_data)} patients loaded in {time.time() - t_load:.1f}s')
print(f'Precomputed sequences held in RAM: {mem_mb:.0f} MB')

## 4 — Model Architecture

Same residual block as the baseline, with `n_blocks = 3`.
RF = 1 + 2(3−1)(2³−1) = 29 — still ~3× the lookback of 10.

In [ ]:
def residual_block(x, filters, kernel_size, dilation_rate, dropout_rate):
    out = Conv1D(filters, kernel_size, dilation_rate=dilation_rate, padding='causal')(x)
    out = BatchNormalization()(out)
    out = Activation('relu')(out)
    out = Dropout(dropout_rate)(out)
    out = Conv1D(filters, kernel_size, dilation_rate=dilation_rate, padding='causal')(out)
    out = BatchNormalization()(out)
    out = Activation('relu')(out)
    out = Dropout(dropout_rate)(out)
    if x.shape[-1] != filters:
        x = Conv1D(filters, 1)(x)
    return Add()([x, out])


def build_tcn(lookback, output_size, n_blocks, n_filters, kernel_size, dropout_rate, learning_rate):
    """Build a TCN with explicit hyperparameters."""
    inp = Input(shape=(lookback, 1))
    x = inp
    for i in range(n_blocks):
        x = residual_block(x, n_filters, kernel_size, 2 ** i, dropout_rate)
    x = x[:, -1, :]  # last time-step
    x = Dense(n_filters, activation='relu')(x)
    out = Dense(output_size)(x)
    model = Model(inp, out, name=f'TCN_out{output_size}')
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate), loss='mse')
    return model


# Sanity check - parameter count at each filter width
for f in FILTER_OPTIONS:
    m = build_tcn(LOOKBACK, PILOT_HORIZON, FIXED_N_BLOCKS, f, FIXED_KERNEL_SIZE, 0.1, 1e-3)
    print(f'{FIXED_N_BLOCKS} blocks, {f:>3d} filters: {m.count_params():>9,} params')
    del m
tf.keras.backend.clear_session()
print('\nModel architecture defined.')

### 1.1 — Verify the GPU is actually being used

Listing a device is not the same as running on it. This cell puts a real op and a real training step on the GPU and inspects where TensorFlow placed them, then fails loudly if the answer is the CPU.

On the cluster, if this cell raises:

- **`nvidia-smi` works but TF sees nothing** — the CUDA/cuDNN module is not loaded in the job environment. Load it in the submit script *before* the Python process starts (`module load cuda/12.x cudnn`), not from inside the notebook.
- **`is_cuda_build: False`** — the installed TensorFlow is a CPU-only wheel. Install the CUDA-enabled one: `pip install "tensorflow[and-cuda]"`. Note that native-Windows TensorFlow has been CPU-only since 2.11; on Windows the GPU path is WSL2.
- **Job scheduled without a GPU** — the submit script is missing its resource request (`#SBATCH --gres=gpu:1`, or the site equivalent).

Ask the cluster admin for: GPU model, GPUs available per job, wall-clock limit, and the module names for CUDA and cuDNN.

In [ ]:
# Verify placement, not just visibility.
gpu_names = [d.name for d in tf.config.list_logical_devices('GPU')]

if not gpu_names:
    msg = ('No GPU visible to TensorFlow.\n'
           f'  tf.__version__      : {tf.__version__}\n'
           f'  built with CUDA     : {build.get("is_cuda_build")}\n'
           '  Check, in order: the job requested a GPU; the CUDA/cuDNN module is\n'
           '  loaded in the job environment; TensorFlow is the CUDA-enabled wheel\n'
           '  ("pip install tensorflow[and-cuda]"). See the notes above this cell.')
    if REQUIRE_GPU:
        raise RuntimeError(msg)
    print('WARNING - ' + msg)
else:
    # 1. A real op, placed explicitly, with the placement read back.
    with tf.device('/GPU:0'):
        a = tf.random.normal((1024, 1024))
        prod = tf.matmul(a, a)
    print(f'matmul executed on   : {prod.device}')
    assert 'GPU' in prod.device, f'matmul landed on {prod.device}, not a GPU'

    # 2. A real training step on the actual architecture, timed.
    probe = build_tcn(10, 10, 3, 64, 3, 0.1, 1e-3)
    print(f'model weights live on: {probe.weights[0].device}')
    assert 'GPU' in probe.weights[0].device, 'model variables are not on the GPU'

    Xp = np.random.rand(4096, 10, 1).astype('float32')
    yp = np.random.rand(4096, 10).astype('float32')
    probe.fit(Xp, yp, epochs=1, batch_size=128, verbose=0)     # warm up / compile
    t0 = time.time()
    probe.fit(Xp, yp, epochs=3, batch_size=128, verbose=0)
    step_ms = (time.time() - t0) / 3 / (len(Xp) / 128) * 1000

    del probe, Xp, yp
    tf.keras.backend.clear_session()
    gc.collect()

    print(f'GPU(s) in use        : {gpu_names}')
    print(f'Measured             : {step_ms:.2f} ms per training step (batch 128)')
    print()
    # 2xT4 reference: ~1.5-3 ms/step at 64 filters. CPU is typically 20ms+.
    if step_ms > 15:
        print('NOTE: that is slow for a GPU on a model this small. Confirm the job is')
        print('      not sharing the card, and that the data feed is not the bottleneck.')
    else:
        print('Throughput is in the expected range for a GPU.')

## 5 — MOGWO Implementation

Custom implementation following Mirjalili et al. (2016), Section 3 and Fig. 2. Unchanged from the 5-patient notebook.

**Key components:**
- **Archive**: stores non-dominated (Pareto-optimal) solutions
- **Grid mechanism**: divides objective space into cells for diversity
- **Leader selection**: picks α, β, δ from the least-crowded archive cells via roulette wheel
- **Position update**: GWO equations (3.5–3.11) with `a` decreasing 2 → 0

In [ ]:
class MOGWO:
    """
    Multi-Objective Grey Wolf Optimizer.
    Reference: Mirjalili et al. (2016), Expert Systems with Applications 47:106-119.
    """

    def __init__(self, n_wolves, max_iter, n_obj, lb, ub,
                 archive_max=20, n_grids=10, seed=None):
        self.n_wolves = n_wolves
        self.max_iter = max_iter
        self.n_obj = n_obj
        self.lb = np.array(lb, dtype=np.float64)
        self.ub = np.array(ub, dtype=np.float64)
        self.n_dim = len(lb)
        self.archive_max = archive_max
        self.n_grids = n_grids
        if seed is not None:
            np.random.seed(seed)

        # Archive storage
        self.archive_solutions = np.empty((0, self.n_dim))
        self.archive_objectives = np.empty((0, self.n_obj))

        # Grid boundaries (updated dynamically)
        self.grid_min = np.zeros(self.n_obj)
        self.grid_max = np.ones(self.n_obj)

    # Pareto dominance (Def. 1 in paper)
    @staticmethod
    def dominates(obj_a, obj_b):
        """Check if a dominates b (all objectives minimised)."""
        return np.all(obj_a <= obj_b) and np.any(obj_a < obj_b)

    def get_non_dominated(self, objectives):
        """Return indices of non-dominated solutions."""
        n = len(objectives)
        is_dominated = np.zeros(n, dtype=bool)
        for i in range(n):
            if is_dominated[i]:
                continue
            for j in range(n):
                if i == j or is_dominated[j]:
                    continue
                if self.dominates(objectives[j], objectives[i]):
                    is_dominated[i] = True
                    break
        return np.where(~is_dominated)[0]

    # Archive management
    def update_archive(self, new_solutions, new_objectives):
        """Add non-dominated solutions to the archive, remove dominated ones."""
        if len(self.archive_solutions) > 0:
            all_sol = np.vstack([self.archive_solutions, new_solutions])
            all_obj = np.vstack([self.archive_objectives, new_objectives])
        else:
            all_sol = new_solutions.copy()
            all_obj = new_objectives.copy()

        nd_idx = self.get_non_dominated(all_obj)
        self.archive_solutions = all_sol[nd_idx].copy()
        self.archive_objectives = all_obj[nd_idx].copy()

        # Overflow - remove from the most crowded cell
        while len(self.archive_solutions) > self.archive_max:
            cell_map = self._get_cell_map()
            most_crowded = max(cell_map, key=lambda k: len(cell_map[k]))
            remove_idx = np.random.choice(cell_map[most_crowded])
            mask = np.ones(len(self.archive_solutions), dtype=bool)
            mask[remove_idx] = False
            self.archive_solutions = self.archive_solutions[mask]
            self.archive_objectives = self.archive_objectives[mask]

        self._update_grid_bounds()

    # Grid mechanism (Section 3 in paper)
    def _update_grid_bounds(self):
        """Update grid boundaries from the current archive extent."""
        if len(self.archive_objectives) == 0:
            return
        self.grid_min = self.archive_objectives.min(axis=0)
        self.grid_max = self.archive_objectives.max(axis=0)
        margin = (self.grid_max - self.grid_min) * 0.1
        margin = np.where(margin == 0, 0.1, margin)
        self.grid_min -= margin
        self.grid_max += margin

    def _get_grid_indices(self):
        """Get the grid cell index of each archive member."""
        rng = self.grid_max - self.grid_min
        rng = np.where(rng == 0, 1.0, rng)
        normalised = (self.archive_objectives - self.grid_min) / rng
        indices = np.floor(normalised * self.n_grids).astype(int)
        return np.clip(indices, 0, self.n_grids - 1)

    def _get_cell_map(self):
        """Map grid cell -> list of archive member indices."""
        grid_idx = self._get_grid_indices()
        cell_map = {}
        for i, gi in enumerate(grid_idx):
            key = tuple(gi)
            if key not in cell_map:
                cell_map[key] = []
            cell_map[key].append(i)
        return cell_map

    # Leader selection (Eq. 3.12 in paper)
    def select_leader(self, exclude_indices=None):
        """
        Select a leader from the archive via roulette wheel over the
        least-crowded cells. P_i = c / N_i (Eq. 3.12).
        """
        if exclude_indices is None:
            exclude_indices = []

        n_archive = len(self.archive_solutions)
        if n_archive == 0:
            raise ValueError('Archive is empty')

        available = [i for i in range(n_archive) if i not in exclude_indices]
        if len(available) == 0:
            # All excluded - pick any (paper handles archives with < 3 members)
            return np.random.randint(n_archive)

        cell_map = self._get_cell_map()
        probs = np.zeros(n_archive)
        for cell_key, members in cell_map.items():
            n_i = len(members)
            for idx in members:
                probs[idx] = 1.0 / n_i  # P_i = c/N_i with c = 1

        for ei in exclude_indices:
            if ei < n_archive:
                probs[ei] = 0.0

        total = probs.sum()
        if total == 0:
            probs[available] = 1.0
            total = probs.sum()
        probs /= total

        return np.random.choice(n_archive, p=probs)

    # Main optimisation loop (Fig. 2 pseudocode)
    def solve(self, obj_func, verbose=True):
        """
        Run MOGWO.

        Returns
        -------
        archive_solutions : np.ndarray - Pareto-optimal solution vectors
        archive_objectives : np.ndarray - corresponding objective values
        eval_log : list[dict] - full evaluation history
        """
        eval_log = []

        # Step 1: initialise the wolf population randomly
        wolves = np.random.uniform(self.lb, self.ub, (self.n_wolves, self.n_dim))
        objectives = np.zeros((self.n_wolves, self.n_obj))

        if verbose:
            print(f'\n{"="*70}')
            print(f'MOGWO Initialisation - Evaluating {self.n_wolves} wolves')
            print(f'{"="*70}')

        # Step 2: evaluate the initial population
        for i in range(self.n_wolves):
            objectives[i] = obj_func(wolves[i])
            eval_log.append({
                'iteration': 0, 'wolf': i,
                'solution': wolves[i].copy(),
                'objectives': objectives[i].copy(),
            })

        # Step 3: initialise the archive with non-dominated solutions
        self.update_archive(wolves, objectives)

        if verbose:
            print(f'\nArchive after init: {len(self.archive_solutions)} solutions')

        # Step 4: main iteration loop
        for t in range(1, self.max_iter + 1):
            # a decreases linearly 2 -> 0 (exploration -> exploitation)
            a = 2.0 - t * (2.0 / self.max_iter)

            alpha_idx = self.select_leader()
            beta_idx  = self.select_leader(exclude_indices=[alpha_idx])
            delta_idx = self.select_leader(exclude_indices=[alpha_idx, beta_idx])

            X_alpha = self.archive_solutions[alpha_idx]
            X_beta  = self.archive_solutions[beta_idx]
            X_delta = self.archive_solutions[delta_idx]

            if verbose:
                print(f'\n{"="*70}')
                print(f'MOGWO Iteration {t}/{self.max_iter} - a={a:.3f}')
                print(f'{"="*70}')

            # Update each wolf's position (Eq. 3.5-3.11)
            for i in range(self.n_wolves):
                r1, r2 = np.random.random(self.n_dim), np.random.random(self.n_dim)
                A1 = 2.0 * a * r1 - a    # Eq. 3.3
                C1 = 2.0 * r2            # Eq. 3.4

                r1, r2 = np.random.random(self.n_dim), np.random.random(self.n_dim)
                A2 = 2.0 * a * r1 - a
                C2 = 2.0 * r2

                r1, r2 = np.random.random(self.n_dim), np.random.random(self.n_dim)
                A3 = 2.0 * a * r1 - a
                C3 = 2.0 * r2

                # Distance vectors (Eq. 3.5-3.7)
                D_alpha = np.abs(C1 * X_alpha - wolves[i])
                D_beta  = np.abs(C2 * X_beta  - wolves[i])
                D_delta = np.abs(C3 * X_delta - wolves[i])

                # Position candidates (Eq. 3.8-3.10)
                X1 = X_alpha - A1 * D_alpha
                X2 = X_beta  - A2 * D_beta
                X3 = X_delta - A3 * D_delta

                # New position = average (Eq. 3.11)
                wolves[i] = (X1 + X2 + X3) / 3.0
                wolves[i] = np.clip(wolves[i], self.lb, self.ub)

            # Evaluate the updated population
            for i in range(self.n_wolves):
                objectives[i] = obj_func(wolves[i])
                eval_log.append({
                    'iteration': t, 'wolf': i,
                    'solution': wolves[i].copy(),
                    'objectives': objectives[i].copy(),
                })

            self.update_archive(wolves, objectives)

            if verbose:
                print(f'Archive size: {len(self.archive_solutions)}')
                print(f'RMSE  range: [{self.archive_objectives[:,0].min():.6f}, '
                      f'{self.archive_objectives[:,0].max():.6f}]')
                print(f'Time  range: [{self.archive_objectives[:,1].min():.1f}s, '
                      f'{self.archive_objectives[:,1].max():.1f}s]')

        return self.archive_solutions.copy(), self.archive_objectives.copy(), eval_log


print('MOGWO class defined.')

## 6 — Objective Function

**Returns two values:** `[mean_RMSE, mean_time]`
- Train on the train split, early-stop on the val split, evaluate RMSE on the test split
- Time = wall-clock seconds for fit + predict per patient, averaged over the pilot patients

### 6.1 — Resume after a killed job

Every completed evaluation is appended to `mogwo21_eval_cache.json`, keyed by its decoded hyperparameters. If the job is killed by a wall-clock limit, just re-run the notebook top to bottom: the RNG is seeded, so MOGWO regenerates the exact same sequence of wolf positions, each already-evaluated position is served from the cache in milliseconds, and training resumes at the first evaluation that never finished. Nothing is recomputed and the search path is identical to an uninterrupted run.

Delete the cache file to force a genuine fresh search.

The cache also removes duplicate work *within* a run. In the 5-patient run, evaluations 27–30 were the same point in search space evaluated four times (the population had converged), which cost four full evaluations — about 3.7 hours at 21 patients — to re-measure a value already known.

In [ ]:
class EvalCache:
    """On-disk memo of completed evaluations, keyed by decoded hyperparameters.

    Enables (a) exact resume after a killed job and (b) skipping duplicate
    points once the wolf population converges.
    """

    def __init__(self, path):
        self.path = path
        self.store = {}
        if os.path.exists(path):
            with open(path, 'r') as f:
                self.store = json.load(f)
            print(f'Loaded {len(self.store)} cached evaluations from {path}')
        else:
            print(f'No cache found - starting a fresh search (cache -> {path})')

    @staticmethod
    def make_key(n_filters, dropout, lr):
        return f'{n_filters}|{dropout:.6f}|{lr:.8e}'

    def get(self, key):
        return self.store.get(key)

    def put(self, key, record):
        self.store[key] = record
        tmp = self.path + '.tmp'
        with open(tmp, 'w') as f:
            json.dump(self.store, f, indent=1)
        os.replace(tmp, self.path)   # atomic - a kill mid-write cannot corrupt it

    def __len__(self):
        return len(self.store)


eval_cache = EvalCache(CACHE_PATH)
print(f'Cache entries: {len(eval_cache)}')

In [ ]:
# Full log of every evaluation MOGWO requests (cached ones included)
mogwo_eval_log = []

# Wall-clock accounting for the ETA
_timing = {'computed_evals': 0, 'computed_seconds': 0.0, 't0': None}


def decode_solution(solution):
    """Solution vector -> hyperparameters."""
    filter_idx = int(np.clip(np.round(solution[0]), 0, len(FILTER_OPTIONS) - 1))
    n_filters  = FILTER_OPTIONS[filter_idx]
    dropout    = float(np.clip(solution[1], 0.05, 0.40))
    lr         = float(10 ** np.clip(solution[2], -4, -2))
    return n_filters, dropout, lr


def evaluate_patient(pid, n_filters, dropout, lr):
    """Train + evaluate one patient. Returns (rmse, elapsed_seconds)."""
    tf.keras.backend.clear_session()
    gc.collect()

    d = patient_data[pid]
    model = build_tcn(LOOKBACK, PILOT_HORIZON,
                      FIXED_N_BLOCKS, n_filters, FIXED_KERNEL_SIZE,
                      dropout, lr)

    t0 = time.time()
    model.fit(
        d['X_tr'], d['y_tr'],
        validation_data=(d['X_vl'], d['y_vl']),
        epochs=EVAL_EPOCHS,
        batch_size=FIXED_BATCH_SIZE,
        callbacks=[
            EarlyStopping(monitor='val_loss', patience=EVAL_PATIENCE,
                          restore_best_weights=True),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                              patience=10, min_lr=1e-6),
        ],
        verbose=0
    )
    preds = model.predict(d['X_te'], batch_size=2048, verbose=0)
    elapsed = time.time() - t0

    rmse = float(np.sqrt(mean_squared_error(d['y_te'].flatten(), preds.flatten())))

    del model
    gc.collect()
    return rmse, elapsed


def objective(solution):
    """MOGWO objective: decode -> train the TCN on every pilot patient
    -> return [mean_RMSE, mean_time]."""
    n_filters, dropout, lr = decode_solution(solution)
    key = EvalCache.make_key(n_filters, dropout, lr)

    eval_id = len(mogwo_eval_log) + 1
    header = (f'  [Eval {eval_id:3d}/{N_EVALS_PLANNED}] '
              f'filters={n_filters:>3d}, dropout={dropout:.3f}, lr={lr:.6f}')

    cached = eval_cache.get(key)
    if cached is not None:
        print(f'{header} -> RMSE={cached["mean_rmse"]:.6f}, '
              f'Time={cached["mean_time"]:.1f}s  [cached]')
        mogwo_eval_log.append({**cached, 'eval': eval_id, 'cached': True})
        return np.array([cached['mean_rmse'], cached['mean_time']])

    print(header, flush=True)
    t_eval = time.time()
    rmse_scores, time_scores = [], []

    for pid in tqdm(PILOT_PATIENTS, desc=f'    eval {eval_id}', leave=False):
        rmse, elapsed = evaluate_patient(pid, n_filters, dropout, lr)
        rmse_scores.append(rmse)
        time_scores.append(elapsed)

    mean_rmse = float(np.mean(rmse_scores))
    mean_time = float(np.mean(time_scores))
    eval_wall = time.time() - t_eval

    # ETA from evaluations actually computed in this session
    _timing['computed_evals']   += 1
    _timing['computed_seconds'] += eval_wall
    mean_eval = _timing['computed_seconds'] / _timing['computed_evals']
    remaining = max(0, N_EVALS_PLANNED - eval_id)

    print(f'    -> RMSE={mean_rmse:.6f} (sd {np.std(rmse_scores):.6f}), '
          f'Time={mean_time:.1f}s/patient, eval took {eval_wall/60:.1f} min')
    print(f'    -> {remaining} evals left, ETA {remaining * mean_eval / 3600:.1f} h '
          f'at {mean_eval/60:.1f} min/eval')

    record = {
        'n_filters': n_filters,
        'dropout': round(dropout, 6),
        'lr': lr,
        'mean_rmse': mean_rmse,
        'std_rmse': float(np.std(rmse_scores)),
        'mean_time': round(mean_time, 2),
        'eval_wall_s': round(eval_wall, 1),
        'per_patient_rmse': {p: round(r, 6) for p, r in zip(PILOT_PATIENTS, rmse_scores)},
    }
    eval_cache.put(key, record)                       # persisted before returning
    mogwo_eval_log.append({**record, 'eval': eval_id, 'cached': False})

    return np.array([mean_rmse, mean_time])


print('Objective function defined.')
print(f'Each evaluation trains on {len(PILOT_PATIENTS)} patients x H={PILOT_HORIZON}')
print(f'Fixed: n_blocks={FIXED_N_BLOCKS}, kernel={FIXED_KERNEL_SIZE}, batch={FIXED_BATCH_SIZE}')
print(f'Returns: [mean_RMSE, mean_time]')

## 7 — Run the MOGWO Search

**Budget:** 5 wolves × (1 init + 5 iterations) = 30 evaluations
**Expected:** ≈ 27.6 h on 2×T4, ≈ 13–15 h on one A100/H100

Safe to re-run after a crash — see Section 6.1.

In [ ]:
print(f'MOGWO config      : {POP_SIZE} wolves x {MAX_ITER} iterations')
print(f'Planned evals     : {N_EVALS_PLANNED}')
print(f'Already cached    : {len(eval_cache)}')
print(f'Archive max       : {ARCHIVE_MAX}, grid divisions: {N_GRIDS}')
print(f'Pilot patients    : {len(PILOT_PATIENTS)}')
print(f'Estimated runtime : {est_hours:.1f} h (2xT4 reference, minus anything cached)')
print()

# Re-seed so a resumed run reproduces the identical search trajectory
np.random.seed(SEED)
tf.random.set_seed(SEED)

optimizer = MOGWO(
    n_wolves=POP_SIZE,
    max_iter=MAX_ITER,
    n_obj=2,           # RMSE, time
    lb=LB,
    ub=UB,
    archive_max=ARCHIVE_MAX,
    n_grids=N_GRIDS,
    seed=SEED,
)

t_start = time.time()
archive_solutions, archive_objectives, eval_history = optimizer.solve(
    obj_func=objective,
    verbose=True,
)
total_time = time.time() - t_start

n_computed = sum(1 for e in mogwo_eval_log if not e['cached'])
print(f'\n{"="*70}')
print('MOGWO Search Complete')
print(f'  Wall-clock time   : {total_time/60:.1f} min ({total_time/3600:.2f} h)')
print(f'  Total evaluations : {len(mogwo_eval_log)}')
print(f'  Computed now      : {n_computed}')
print(f'  Served from cache : {len(mogwo_eval_log) - n_computed}')
print(f'  Archive size      : {len(archive_solutions)}')
print(f'{"="*70}')

## 8 — Analysis & Selection

In [ ]:
# 8.1 Decode the Pareto archive
archive_decoded = []
for sol, obj in zip(archive_solutions, archive_objectives):
    n_filters, dropout, lr = decode_solution(sol)
    archive_decoded.append({
        'n_filters': n_filters,
        'dropout': round(dropout, 4),
        'lr': round(lr, 8),
        'RMSE': round(float(obj[0]), 6),
        'Time_s': round(float(obj[1]), 2),
    })

df_archive = pd.DataFrame(archive_decoded).sort_values('RMSE').reset_index(drop=True)
print(f'Pareto archive - {len(df_archive)} non-dominated solutions:\n')
display(df_archive)

### 8.2 — Weighted selection: 0.7 × RMSE + 0.3 × time

The two objectives are on wildly different scales — RMSE ≈ 0.027, time ≈ 158 s, four orders of magnitude apart. A raw `0.7*RMSE + 0.3*time` would be 99.99 % driven by the time term and the weights would mean nothing. Both objectives are therefore min-max normalised to [0, 1] across the archive first, so the weights carry their intended meaning:

$$\text{score} = 0.7 \cdot \frac{\text{RMSE} - \text{RMSE}_{\min}}{\text{RMSE}_{\max} - \text{RMSE}_{\min}} + 0.3 \cdot \frac{t - t_{\min}}{t_{\max} - t_{\min}}$$

**Why this is applied after the search rather than inside the objective.** Collapsing two objectives into one weighted number *is* scalarization: it turns MOGWO back into ordinary single-objective GWO. There would be no Pareto archive, no dominance test, no grid-based leader selection — the entire multi-objective contribution of the work would disappear, and the notebook would be the GWO search you already ran with a different loss.

Applying the weights afterwards costs nothing and gives both: the full Pareto front for the paper, plus the single configuration your TA's 70/30 preference points at. For a convex Pareto front the minimiser of the weighted sum lies on the front anyway, so the result is the same point a scalarized search would have converged to — obtained without discarding the front.

Changing the weights later needs no re-run; just edit `W_RMSE`/`W_TIME` and re-execute this cell.

In [ ]:
def weighted_score(rmse, times, w_rmse=W_RMSE, w_time=W_TIME):
    """Min-max normalise each objective to [0,1], then weight.

    Normalisation is what makes the weights meaningful - RMSE ~0.027 and time
    ~158s are four orders of magnitude apart in raw units.
    """
    rmse = np.asarray(rmse, dtype=float)
    times = np.asarray(times, dtype=float)
    r_rng = rmse.max() - rmse.min()
    t_rng = times.max() - times.min()
    r_norm = (rmse - rmse.min()) / r_rng if r_rng > 1e-12 else np.zeros_like(rmse)
    t_norm = (times - times.min()) / t_rng if t_rng > 1e-12 else np.zeros_like(times)
    return w_rmse * r_norm + w_time * t_norm, r_norm, t_norm


scores, r_norm, t_norm = weighted_score(df_archive['RMSE'], df_archive['Time_s'])
df_archive['RMSE_norm']  = np.round(r_norm, 4)
df_archive['Time_norm']  = np.round(t_norm, 4)
df_archive['Score_70_30'] = np.round(scores, 4)

best_idx = int(np.argmin(scores))
best = df_archive.loc[best_idx]

print(f'Archive ranked by {W_RMSE:.0%} RMSE / {W_TIME:.0%} time:\n')
display(df_archive.sort_values('Score_70_30').reset_index(drop=True))

print(f'\n{"="*70}')
print(f'SELECTED CONFIGURATION  ({W_RMSE:.0%} RMSE / {W_TIME:.0%} time)')
print(f'{"="*70}')
print(f'  n_filters   = {int(best["n_filters"])}')
print(f'  dropout     = {best["dropout"]}')
print(f'  lr          = {best["lr"]}')
print(f'  n_blocks    = {FIXED_N_BLOCKS}   (fixed)')
print(f'  kernel_size = {FIXED_KERNEL_SIZE}   (fixed)')
print(f'  batch_size  = {FIXED_BATCH_SIZE} (fixed)')
print(f'\n  Pilot RMSE  = {best["RMSE"]:.6f}   (normalised {best["RMSE_norm"]:.3f})')
print(f'  Pilot time  = {best["Time_s"]:.1f}s/patient   (normalised {best["Time_norm"]:.3f})')
print(f'  Score       = {best["Score_70_30"]:.4f}')
print(f'{"="*70}')

# Cross-check: the same rule applied to every evaluation, not just the archive.
# These should agree - if they do not, the front has a non-convex region.
df_evals = pd.DataFrame(mogwo_eval_log)
all_scores, _, _ = weighted_score(df_evals['mean_rmse'], df_evals['mean_time'])
alt = df_evals.loc[int(np.argmin(all_scores))]
same = (int(alt['n_filters']) == int(best['n_filters'])
        and abs(alt['dropout'] - best['dropout']) < 1e-3)
print(f'\nCross-check over all {len(df_evals)} evaluations: '
      f'filters={int(alt["n_filters"])}, dropout={alt["dropout"]:.3f}, lr={alt["lr"]:.6f} '
      f'-> {"agrees" if same else "DIFFERS - inspect the front for a non-convex region"}')

In [ ]:
# 8.3 Full evaluation log
df_evals_show = df_evals[['eval', 'n_filters', 'dropout', 'lr',
                          'mean_rmse', 'std_rmse', 'mean_time', 'cached']]
print(f'All {len(df_evals)} evaluations, best RMSE first:\n')
display(df_evals_show.sort_values('mean_rmse').reset_index(drop=True))

In [ ]:
# 8.4 Pareto front, with the 70/30 pick marked
fig, ax = plt.subplots(figsize=(11, 7))

ax.scatter(df_evals['mean_time'], df_evals['mean_rmse'],
           c='#BDBDBD', s=40, alpha=0.5, label='All evaluations', zorder=1)

colors_map = {32: '#4CAF50', 64: '#2196F3', 128: '#FF9800', 256: '#F44336'}
for _, row in df_archive.iterrows():
    ax.scatter(row['Time_s'], row['RMSE'],
               c=colors_map.get(row['n_filters'], '#000000'),
               s=120, edgecolors='black', linewidths=1.5, zorder=3)

df_front = df_archive.sort_values('Time_s')
ax.plot(df_front['Time_s'], df_front['RMSE'], '--', color='#333333',
        linewidth=1.5, alpha=0.7, zorder=2)

# The selected point
ax.scatter(best['Time_s'], best['RMSE'], marker='*', s=700,
           facecolor='#FFD54F', edgecolors='black', linewidths=1.8, zorder=5,
           label=f'Selected ({W_RMSE:.0%} RMSE / {W_TIME:.0%} time)')

for filt, color in colors_map.items():
    ax.scatter([], [], c=color, s=80, edgecolors='black',
               linewidths=1, label=f'{filt} filters')

for _, row in df_archive.iterrows():
    ax.annotate(f'{int(row["n_filters"])}f, d={row["dropout"]:.2f}\nlr={row["lr"]:.1e}',
                (row['Time_s'], row['RMSE']),
                textcoords='offset points', xytext=(10, 5),
                fontsize=7, alpha=0.8)

ax.set_xlabel('Mean training time per patient (s)', fontsize=12)
ax.set_ylabel('Mean RMSE', fontsize=12)
ax.set_title(f'MOGWO Pareto front - RMSE vs training time '
             f'({len(PILOT_PATIENTS)} patients, H={PILOT_HORIZON})', fontsize=14)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'mogwo21_pareto_front.png'), dpi=200, bbox_inches='tight')
plt.show()

print('Points on the dashed line are non-dominated: no other solution beats them on BOTH objectives.')
print(f'The star is the {W_RMSE:.0%}/{W_TIME:.0%} weighted pick - carry it into the final evaluation notebook.')

In [ ]:
# 8.5 Convergence
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

iter_stats = []
best_rmse_so_far = float('inf')
for entry in eval_history:
    rmse_val = float(entry['objectives'][0])
    best_rmse_so_far = min(best_rmse_so_far, rmse_val)
    iter_stats.append({
        'eval_num': len(iter_stats) + 1,
        'iteration': entry['iteration'],
        'rmse': rmse_val,
        'time': float(entry['objectives'][1]),
        'best_rmse': best_rmse_so_far,
    })
df_stats = pd.DataFrame(iter_stats)

ax = axes[0]
ax.scatter(df_stats['eval_num'], df_stats['rmse'], alpha=0.6, s=40, c='steelblue')
ax.plot(df_stats['eval_num'], df_stats['best_rmse'], 'r-', linewidth=2,
        label='Best RMSE so far')
ax.set_xlabel('Evaluation #')
ax.set_ylabel('Mean RMSE')
ax.set_title('RMSE convergence')
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
sc = ax.scatter(df_stats['time'], df_stats['rmse'], c=df_stats['iteration'],
                cmap='viridis', s=50, alpha=0.8, edgecolors='white', linewidths=0.5)
plt.colorbar(sc, ax=ax, label='Iteration')
ax.set_xlabel('Training time (s)')
ax.set_ylabel('Mean RMSE')
ax.set_title('All evaluations in objective space')
ax.grid(alpha=0.3)

plt.suptitle(f'MOGWO search progress ({len(PILOT_PATIENTS)} patients)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'mogwo21_convergence.png'), dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# 8.6 Parameter exploration
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, col, label in zip(axes,
                          ['n_filters', 'dropout', 'lr'],
                          ['n_filters', 'dropout', 'learning rate']):
    sc = ax.scatter(df_evals[col], df_evals['mean_rmse'],
                    c=df_evals['mean_time'], cmap='RdYlGn_r', s=60, alpha=0.85)
    ax.set_xlabel(label)
    ax.set_ylabel('Mean RMSE')
    ax.set_title(f'RMSE vs {label}\n(colour = time)')
    ax.grid(alpha=0.3)
    if col == 'lr':
        ax.set_xscale('log')

plt.colorbar(sc, ax=axes[-1], label='Mean time (s)')
plt.suptitle('Parameter exploration', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'mogwo21_param_distributions.png'), dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# 8.7 Measurement noise: is the search resolving signal or luck?
#
# Per-evaluation spread across patients is not the quantity of interest - what
# matters is the standard error of the MEAN, since the mean is what MOGWO
# compares. Compare it against the RMSE spread of the Pareto front.
sem = df_evals['std_rmse'] / np.sqrt(len(PILOT_PATIENTS))
front_spread = df_archive['RMSE'].max() - df_archive['RMSE'].min()

print(f'Pilot patients                    : {len(PILOT_PATIENTS)}')
print(f'Median across-patient SD of RMSE  : {df_evals["std_rmse"].median():.6f}')
print(f'Median standard error of the mean : {sem.median():.6f}')
print(f'Pareto front RMSE spread          : {front_spread:.6f}')
ratio = front_spread / sem.median() if sem.median() > 0 else float('inf')
print(f'Signal-to-noise (spread / SEM)    : {ratio:.1f}x')
print()
if ratio >= 3:
    print('Front spread comfortably exceeds measurement noise - differences between')
    print('archive members are real, and ranking them is meaningful.')
elif ratio >= 1:
    print('Front spread exceeds noise but not by much. Treat the ranking WITHIN the')
    print('archive as indicative; the selected point is sound, near-neighbours are ties.')
else:
    print('Noise exceeds the front spread - the archive members are statistically')
    print('indistinguishable. Report the selected config, but do not claim it beats')
    print('its neighbours on the front. More patients or seed-averaging would be needed.')
print()
print('Reference: the 5-patient run had ~0.0021 seed-to-seed spread against a')
print('~0.0015 front spread, i.e. a ratio below 1. That is what this run fixes.')

In [ ]:
# 8.8 Comparison with the earlier searches
print('Single-objective GWO (4 blocks, 5-patient pilot):')
print('  n_filters=256, dropout=0.109, lr=0.002066  -> pilot RMSE=0.026152')
print()
print('MOGWO, 5-patient pilot (3 blocks):')
print('  best observed RMSE=0.026148 at filters=128, dropout=0.131, lr=0.004510')
print('  30 evaluations in 6.8 h on 2xT4')
print()
print(f'MOGWO, {len(PILOT_PATIENTS)}-patient pilot (3 blocks) - this run:')
for i, (_, row) in enumerate(df_archive.sort_values('Score_70_30').iterrows()):
    mark = '  <-- selected' if i == 0 else ''
    print(f'  [{i+1}] filters={int(row["n_filters"]):>3d}, dropout={row["dropout"]:.3f}, '
          f'lr={row["lr"]:.6f} -> RMSE={row["RMSE"]:.6f}, '
          f'Time={row["Time_s"]:.1f}s, score={row["Score_70_30"]:.4f}{mark}')
print()
print('Note: pilot RMSE values are comparable across runs ONLY where the patient')
print('set matches. A 21-patient mean includes harder records than the 5-patient')
print('mean, so a higher number here is not a regression.')

## 9 — Save Results

Everything the final-evaluation notebook needs is written to `mogwo21_output/`.

In [ ]:
# Save the Pareto archive, the full evaluation log, and a machine-readable summary
archive_path = os.path.join(OUT_DIR, 'mogwo21_pareto_archive.csv')
df_archive.to_csv(archive_path, index=False)
print(f'Pareto archive     -> {archive_path} ({len(df_archive)} solutions)')

evals_path = os.path.join(OUT_DIR, 'mogwo21_all_evaluations.csv')
df_evals.to_csv(evals_path, index=False)
print(f'All evaluations    -> {evals_path} ({len(df_evals)} evals)')

summary = {
    'optimiser': 'MOGWO (Mirjalili et al. 2016)',
    'objectives': ['mean_RMSE', 'mean_training_time_s'],
    'pilot_patients': PILOT_PATIENTS,
    'holdout_patients': HOLDOUT_PATIENTS,
    'horizon': PILOT_HORIZON,
    'pop_size': POP_SIZE,
    'max_iter': MAX_ITER,
    'total_evaluations': len(df_evals),
    'wall_clock_hours': round(total_time / 3600, 3),
    'fixed': {
        'n_blocks': FIXED_N_BLOCKS,
        'kernel_size': FIXED_KERNEL_SIZE,
        'batch_size': FIXED_BATCH_SIZE,
        'lookback': LOOKBACK,
        'epochs': EVAL_EPOCHS,
        'patience': EVAL_PATIENCE,
    },
    'selection_weights': {'rmse': W_RMSE, 'time': W_TIME},
    'selected': {
        'n_filters': int(best['n_filters']),
        'dropout': float(best['dropout']),
        'lr': float(best['lr']),
        'pilot_rmse': float(best['RMSE']),
        'pilot_time_s': float(best['Time_s']),
        'score': float(best['Score_70_30']),
    },
    'seed': SEED,
}
summary_path = os.path.join(OUT_DIR, 'mogwo21_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Summary            -> {summary_path}')

print(f'\n{"="*70}')
print('MOGWO SEARCH COMPLETE - SUMMARY')
print(f'{"="*70}')
print(f'Optimiser      : MOGWO (Mirjalili et al. 2016)')
print(f'Objectives     : minimize RMSE, minimize training time')
print(f'Search space   : 3 params (n_filters, dropout, lr)')
print(f'Fixed          : n_blocks={FIXED_N_BLOCKS}, kernel={FIXED_KERNEL_SIZE}, batch={FIXED_BATCH_SIZE}')
print(f'Pilot          : {len(PILOT_PATIENTS)} patients, H={PILOT_HORIZON}')
print(f'Population     : {POP_SIZE} wolves x {MAX_ITER} iterations')
print(f'Total evals    : {len(df_evals)}')
print(f'Wall-clock     : {total_time/60:.1f} min ({total_time/3600:.2f} h)')
print(f'Archive size   : {len(df_archive)} Pareto-optimal solutions')
print(f'  Best RMSE    : {df_archive["RMSE"].min():.6f} '
      f'(time={df_archive.loc[df_archive["RMSE"].idxmin(), "Time_s"]:.1f}s)')
print(f'  Fastest      : {df_archive["Time_s"].min():.1f}s '
      f'(RMSE={df_archive.loc[df_archive["Time_s"].idxmin(), "RMSE"]:.6f})')
print(f'\nPaste into the final-evaluation notebook:')
print(f'  NUM_FILTERS   = {int(best["n_filters"])}')
print(f'  DROPOUT_RATE  = {best["dropout"]}')
print(f'  LEARNING_RATE = {best["lr"]}')
print(f'  NUM_BLOCKS    = {FIXED_N_BLOCKS}')
print(f'  KERNEL_SIZE   = {FIXED_KERNEL_SIZE}')
print(f'  BATCH_SIZE    = {FIXED_BATCH_SIZE}')
print(f'\nFiles in {OUT_DIR}/:')
for fn in sorted(os.listdir(OUT_DIR)):
    print(f'  {fn}')
print(f'{"="*70}')